In [ ]:
import os # Interoperable file paths
import pathlib # Find the home folder
import rioxarray as rxr # Work with geospatial raster data

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score

rasterio, scipy.ndimage

import xarray as xr
import numpy as np

import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.pyplot as plt

In [ ]:
# Load NEON Data: NEON hyperspectral image (GeoTIFF or HDF5 converted to a raster-like array).

In [ ]:
# Tiff files
# "C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\NEON_D17_SOAP_DP3_298000_4100000_CWC_burned.tif"
# "C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\NEON_D17_SOAP_DP3_298000_4101000_CWC_unburned.tif"

In [ ]:
# For reflectance data, simple averaging is often a good approach for downsampling. 
# Load NEON data (assuming it's already in a rioxarray DataArray)
# Replace with your actual NEON data loading
neon_data = rxr.open_rasterio("path/to/neon_reflectance.tif")
# Or if from HDF5, load using h5py and then create a DataArray with CRS and transform

# Get EMIT resolution (60m)
emit_resolution = 60

# Define target CRS (assuming both are in the same UTM zone, for example)
# Ensure NEON_data has a CRS set correctly
# neon_data = neon_data.rio.write_crs("EPSG:XXXX", inplace=True) # Replace XXXX with appropriate EPSG code

# Resample NEON data to 60m resolution
# 'resampling=rasterio.enums.Resampling.average' is often good for reflectance
neon_resampled = neon_data.rio.reproject(
    dst_crs=neon_data.rio.crs, # Keep the same CRS
    resolution=emit_resolution,
    resampling=rxr.raster_array.Resampling.average # Or .mode, .nearest, etc.
)

# You might need to align extents more precisely if the areas don't perfectly overlap
# This could involve cropping or padding after resampling, or explicitly defining
# the target bounds for reproject

# Spectral Resampling/Band Alignment 
Find Overlapping Bands: Identify the spectral bands in NEON that correspond to EMIT's bands.

Aggregate NEON Bands: If an EMIT band covers a range of NEON bands, you can average or sum the NEON bands within that range to create a comparable band for NEON.

Resampling Libraries: While rasterio focuses on spatial resampling, for spectral resampling (if not a direct band-to-band match), you'll primarily be working with the spectral dimension of your xarray.DataArray or NumPy array. You might write custom functions or use interpolation.

In [ ]:
# Assuming neon_resampled has a 'wavelength' dimension
# and emit_bands is a dictionary or list of EMIT band ranges/centers

# Example EMIT band (conceptual)
emit_band_center_example = 550 # nm
emit_band_width_example = 7.5 # nm

# Find NEON bands within this range
neon_wavelengths = neon_resampled["wavelength"].values
# Simple example: find bands within +/- half the EMIT band width
matching_neon_bands = neon_resampled.sel(
    wavelength=(
        (neon_wavelengths >= (emit_band_center_example - emit_band_width_example / 2)) &
        (neon_wavelengths <= (emit_band_center_example + emit_band_width_example / 2))
    )
)

if len(matching_neon_bands["wavelength"]) > 0:
    # Average the matching NEON bands
    neon_aggregated_band = matching_neon_bands.mean(dim="wavelength")
else:
    # Handle cases where no NEON band directly aligns (e.g., interpolate or skip)
    pass

CWC and fit a model steps
1. "Target" Variable (y) - Agreement/Disagreement between NEON and EMIT: This is a more advanced classification where you might try to predict when NEON and EMIT CWC values agree or disagree beyond a certain threshold.
y could be a binary variable: 0 for "CWC difference within tolerance" and 1 for "CWC difference outside tolerance."
Crucially: DecisionTreeClassifier is for Classification. If you want to predict a continuous CWC value, you would use sklearn.tree.DecisionTreeRegressor. The principles of splitting data, cross-validation, and plotting are similar.

2. What are Your "Features" (X) - Your features (X) will be the input variables used by the Decision Tree to make its predictions. This is where your processed NEON and EMIT data come in.
Canopy Water Content (CWC): NEON_CWC (the CWC calculated from NEON data at 60m resolution)

3. Preparing Your Data for scikit-learn:

Your data needs to be in a tabular format (like a Pandas DataFrame) where each row represents a spatially matched pixel (or observation), and each column is a feature or the target variable.


In [ ]:

# --- 1. Load your spatially matched and CWC-calculated data ---
# This is a conceptual representation.
# You'd load your actual NEON/EMIT CWC and other spectral data here.
# Assuming you have a way to extract pixel values and align them.

# Example: Create a dummy DataFrame (replace with your actual data loading)
# Each row is a pixel, columns are features and the target.
num_pixels = 1000
data = {
    'NEON_CWC': np.random.rand(num_pixels) * 0.5 + 0.1, # Simulated CWC values
    'EMIT_CWC': np.random.rand(num_pixels) * 0.6 + 0.05,
    'NEON_Band_500nm': np.random.rand(num_pixels),
    'EMIT_Band_550nm': np.random.rand(num_pixels),
    # Add other bands/indices as features
    # 'Vegetation_Type': np.random.randint(0, 3, num_pixels) # Example target for classification
    'Drought_Stress_Level': np.random.randint(0, 3, num_pixels) # Example target for classification
}
df = pd.DataFrame(data)

# --- 2. Define Features (X) and Target (y) ---
# X will be your independent variables (inputs to the model)
# y will be your dependent variable (what you want to predict)

features = ['NEON_CWC', 'EMIT_CWC', 'NEON_Band_500nm', 'EMIT_Band_550nm']
X = df[features]
y = df['Drought_Stress_Level'] # Or 'Vegetation_Type', etc.

# --- 3. Split Data into Training and Testing Sets ---
# This is crucial for evaluating model performance on unseen data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # stratify is good for imbalanced classes
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# --- 4. Fit the Decision Tree Model ---
# Initialize the classifier (or regressor)
clf = DecisionTreeClassifier(max_depth=5, random_state=42) # Limit depth to prevent overfitting

# Fit the model to the training data
clf.fit(X_train, y_train)

print("Model fitting complete!")

# --- 5. Evaluate the Model (using cross_val_score for robustness) ---
# Cross-validation provides a more reliable estimate of model performance
# by training and testing on different subsets of the data multiple times.
cv_scores = cross_val_score(clf, X, y, cv=5) # 5-fold cross-validation
print(f"\nCross-validation scores (accuracy): {cv_scores}")
print(f"Mean cross-validation accuracy: {np.mean(cv_scores):.2f}")

# You can also evaluate on the test set directly (after initial fitting)
from sklearn.metrics import accuracy_score, classification_report
y_pred = clf.predict(X_test)
print(f"\nAccuracy on test set: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report on test set:")
print(classification_report(y_test, y_pred))


# --- 6. Visualize the Decision Tree ---
# This helps interpret the rules learned by the tree.
plt.figure(figsize=(20, 10))
plot_tree(
    clf,
    feature_names=features,
    class_names=[str(c) for c in clf.classes_], # Convert class names to strings if needed
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree for CWC Comparison")
plt.show()

# --- 7. Interpret the Tree / Feature Importance ---
# You can see which features were most influential in the decision-making process.
feature_importances = pd.DataFrame({'feature': features, 'importance': clf.feature_importances_})
feature_importances = feature_importances.sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(feature_importances)

Defining the Target Variable (y): This is the most critical difference.
the decision tree in a more exploratory way to understand splits? If you are just comparing NEON CWC to EMIT CWC, a decision tree might not be the most direct approach unless you categorize the difference between them.
Overfitting: Decision trees can easily overfit, especially with high-dimensional hyperspectral data.

max_depth: Limit the maximum depth of the tree (clf = DecisionTreeClassifier(max_depth=5)).

min_samples_leaf: Set a minimum number of samples required to be at a leaf node.

ccp_alpha: Use cost-complexity pruning (as shown in some scikit-learn examples) to find an optimal pruning parameter.

Cross-Validation (cross_val_score): Always use cross-validation to get a more robust estimate of your model's performance and to help with hyperparameter tuning.

Interpreting the Tree: The plot_tree function is invaluable for understanding the rules the model learned. Look at which features are used at the top of the tree (these are the most important splits) and what thresholds are being applied.